In [1]:
# libraries
import os
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from nltk.stem import PorterStemmer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# dataset path and its folders
dataset_folder_path = "sorted_data_acl"
print(os.listdir(dataset_folder_path))

['books', 'dvd', 'electronics', 'kitchen_&_housewares']


In [3]:
# load positive and negative reviews
review_data = []
for domain in os.listdir(dataset_folder_path):
    domain_path = os.path.join(dataset_folder_path, domain)
    if not os.path.isdir(domain_path):
        continue
    # Load positive reviews
    positive_file = os.path.join(domain_path, "positive.review")
    if os.path.exists(positive_file):
        with open(positive_file, "r", encoding="utf-8") as file:
            for line in file:
                line = line.strip()

                if line:
                    review_data.append({
                        "domain": domain,
                        "sentiment": "positive",
                        "review": line
                    })
    # Load negative reviews
    negative_file = os.path.join(domain_path, "negative.review")
    if os.path.exists(negative_file):
        with open(negative_file, "r", encoding="utf-8") as file:
            for line in file:
                line = line.strip()

                if line:
                    review_data.append({
                        "domain": domain,
                        "sentiment": "negative",
                        "review": line
                    })
sentiment_data = pd.DataFrame(review_data)

In [4]:
# display data
sentiment_data.head()

,domain,sentiment,review
0,books,positive,<review>
1,books,positive,<unique_id>
2,books,positive,0785758968:one_of_the_best_crichton_novels:jos...
3,books,positive,</unique_id>
4,books,positive,<asin>


In [5]:
# no of rows and columns
print("Number of rows:", sentiment_data.shape[0])
print("Number of columns:", sentiment_data.shape[1])

Number of rows: 288221
Number of columns: 3


In [6]:
# shape of the dataset
sentiment_data.shape

(288221, 3)

In [7]:
# column names
sentiment_data.columns.tolist()

['domain', 'sentiment', 'review']

In [8]:
# data types
sentiment_data.dtypes

domain       str
sentiment    str
review       str
dtype: object

In [9]:
# checking missing values
sentiment_data.isnull().sum()

domain       0
sentiment    0
review       0
dtype: int64

In [10]:
# checking duplicate records
sentiment_data.duplicated().sum()

np.int64(221701)

In [11]:
# distribution of positive and negative reviews
sentiment_counts = sentiment_data["sentiment"].value_counts()
sentiment_counts

sentiment
positive    144246
negative    143975
Name: count, dtype: int64

In [12]:
# removing duplicate records
sentiment_data = sentiment_data.drop_duplicates().reset_index(drop=True)
sentiment_data.shape

(66520, 3)

In [13]:
# converting review text to lowercase
sentiment_data["review"] = sentiment_data["review"].astype(str).str.lower()
sentiment_data[["review", "sentiment"]].head()

,review,sentiment
0,<review>,positive
1,<unique_id>,positive
2,0785758968:one_of_the_best_crichton_novels:jos...,positive
3,</unique_id>,positive
4,<asin>,positive


In [14]:
# removing HTML tags
sentiment_data["review"] = sentiment_data["review"].apply(lambda text: re.sub(r"<[^>]+>", " ", text))
sentiment_data.head(10)

,domain,sentiment,review
0,books,positive,
1,books,positive,
2,books,positive,0785758968:one_of_the_best_crichton_novels:jos...
3,books,positive,
4,books,positive,
5,books,positive,0785758968
6,books,positive,
7,books,positive,
8,books,positive,sphere: books: michael crichton
9,books,positive,


In [15]:
# Remove punctuation, numbers and special characters
sentiment_data["review"] = sentiment_data["review"].apply(lambda text: re.sub(r"[^a-zA-Z\s]", " ", text))
sentiment_data.head(10)

,domain,sentiment,review
0,books,positive,
1,books,positive,
2,books,positive,one of the best crichton novels jos...
3,books,positive,
4,books,positive,
5,books,positive,
6,books,positive,
7,books,positive,
8,books,positive,sphere books michael crichton
9,books,positive,


In [16]:
# Remove extra spaces
sentiment_data["review"] = sentiment_data["review"].apply(lambda text: re.sub(r"\s+", " ", text).strip())
sentiment_data.head(10)

,domain,sentiment,review
0,books,positive,
1,books,positive,
2,books,positive,one of the best crichton novels joseph m
3,books,positive,
4,books,positive,
5,books,positive,
6,books,positive,
7,books,positive,
8,books,positive,sphere books michael crichton
9,books,positive,


In [17]:
# Remove empty reviews
sentiment_data = sentiment_data[sentiment_data["review"].str.strip() != ""].reset_index(drop=True)
sentiment_data.head(10)

,domain,sentiment,review
0,books,positive,one of the best crichton novels joseph m
1,books,positive,sphere books michael crichton
2,books,positive,books
3,books,positive,of
4,books,positive,one of the best crichton novels
5,books,positive,july
6,books,positive,joseph m
7,books,positive,colorado usa
8,books,positive,sphere by michael crichton is an excellant nov...
9,books,positive,the story revolves around a man named norman j...


In [18]:
# Remove very short reviews
sentiment_data["review_length"] = sentiment_data["review"].apply(lambda text: len(text.split()))
sentiment_data = sentiment_data[sentiment_data["review_length"] >= 3].reset_index(drop=True)
sentiment_data = sentiment_data.drop(columns=["review_length"])
sentiment_data.head(10)

,domain,sentiment,review
0,books,positive,one of the best crichton novels joseph m
1,books,positive,sphere books michael crichton
2,books,positive,one of the best crichton novels
3,books,positive,sphere by michael crichton is an excellant nov...
4,books,positive,the story revolves around a man named norman j...
5,books,positive,this novel does not have the research that som...
6,books,positive,i would strongly recommend this book
7,books,positive,the medicine of the future wafa rashed
8,books,positive,healing from the heart a leading surgeon combi...
9,books,positive,the medicine of the future


In [19]:
# Stop-word removal
stop_words = set(ENGLISH_STOP_WORDS)
important_words = {"not","no","never","nor","neither","none"}
stop_words = stop_words - important_words
def remove_stopwords(text):
    words = text.split()
    filtered_words = [
        word for word in words
        if word not in stop_words
    ]
    return " ".join(filtered_words)
sentiment_data["review"] = sentiment_data["review"].apply(remove_stopwords)
sentiment_data.head(10)

,domain,sentiment,review
0,books,positive,best crichton novels joseph m
1,books,positive,sphere books michael crichton
2,books,positive,best crichton novels
3,books,positive,sphere michael crichton excellant novel certai...
4,books,positive,story revolves man named norman johnson johnso...
5,books,positive,novel does not research crichton novels lot in...
6,books,positive,strongly recommend book
7,books,positive,medicine future wafa rashed
8,books,positive,healing heart leading surgeon combines eastern...
9,books,positive,medicine future


In [20]:
# Stemming
stemmer = PorterStemmer()
def apply_stemming(text):
    words = text.split()
    stemmed_words = [stemmer.stem(word) for word in words]
    return " ".join(stemmed_words)
sentiment_data["review"] = sentiment_data["review"].apply(apply_stemming)
sentiment_data["review"].head()

0                         best crichton novel joseph m
1                         sphere book michael crichton
2                                  best crichton novel
3    sphere michael crichton excel novel certainli ...
4    stori revolv man name norman johnson johnson p...
Name: review, dtype: str

In [21]:
# Tokenize the review text
tokenizer = Tokenizer(num_words=20000,oov_token="<OOV>")
tokenizer.fit_on_texts(sentiment_data["review"])
sequences = tokenizer.texts_to_sequences(sentiment_data["review"])

In [22]:
# Converting reviews into numerical sequences
sequence_lengths = [len(sequence) for sequence in sequences]
padded_sequences = pad_sequences(sequences,maxlen=100,padding="post",truncating="post")

In [23]:
# encosing sentiment lables
label_encoding = {"negative": 0,"positive": 1}
sentiment_data["sentiment_encoded"] = sentiment_data["sentiment"].map(label_encoding)

In [24]:
sentiment_data.head()

,domain,sentiment,review,sentiment_encoded
0,books,positive,best crichton novel joseph m,1
1,books,positive,sphere book michael crichton,1
2,books,positive,best crichton novel,1
3,books,positive,sphere michael crichton excel novel certainli ...,1
4,books,positive,stori revolv man name norman johnson johnson p...,1


In [25]:
# save the cleaned dataset
cleaned_dataset = pd.DataFrame({"domain": sentiment_data["domain"].values,"review": sentiment_data["review"].values,
                                "sequence": [" ".join(map(str, sequence))
        for sequence in padded_sequences],"sentiment": sentiment_data["sentiment_encoded"].values})
cleaned_dataset.to_csv( "amazon_reviews_cleaned_encoded.csv",index=False)